# ECCO-DarwinDiff — Coupled Two-Tracer Recovery Demo

Step 2 of the prototype validation. Notebook 1 showed the differentiable scaffold works on a single-tracer 1D toy. This one extends to a **coupled** nutrient + phytoplankton system, the smallest test that the same scaffold handles inter-tracer coupling and nonlinear reaction terms — the structural feature that matters for real Darwin biogeochemistry, where one parameter influences many tracers through the BGC reaction network.

Equations (NPZ-style, light-limited growth):

$$\frac{\partial N}{\partial t} = \kappa \frac{\partial^2 N}{\partial z^2} - \mu(z)\, P\, f_{\text{light}}(z)\, N$$

$$\frac{\partial P}{\partial t} = \kappa \frac{\partial^2 P}{\partial z^2} + \mu(z)\, P\, f_{\text{light}}(z)\, N - m\, P^2$$

The MLP learns $\mu(z)$, the depth-dependent uptake rate. Recovery has to reproduce **both** observed tracer fields simultaneously.

In [ ]:
import time

import matplotlib.pyplot as plt
import torch

from darwindiff.prototype.coupled import (
    generate_coupled_observations,
    integrate_coupled,
    light_profile,
    train_coupled_recovery,
    true_mu_profile,
)
from darwindiff.prototype.parameter_mlp import ParameterMLP

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}")
print(
    f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}"
)

## 1. Set up the lit-zone column

Domain is 0\u2013200 m \u2014 the photic depth where $\mu(z)$ is identifiable from the observations. Below the photic zone the uptake term $\mu \, P \, f_{\text{light}} \, N$ is essentially zero regardless of $\mu$, so $\mu(z)$ cannot be recovered there. Restricting the domain keeps the inverse problem well-posed.

Light attenuation length is 100 m, so $f_{\text{light}}$ stays above 0.13 even at the deep boundary. Initial nutrient is depth-graded; the plankton seed is small and uniform.

In [ ]:
Nz = 50
z = torch.linspace(0.0, 200.0, Nz)
N0 = 0.5 + 0.5 * (z / 200.0)
P0 = 0.05 * torch.ones(Nz)
f_light = light_profile(z, attenuation_length=100.0)

kappa = 0.05      # vertical diffusion
mortality = 0.3   # quadratic plankton loss
dz = 4.0          # spatial grid spacing (m)
dt = 1.0          # time step
n_steps = 200     # integration length

print(f"Column: {Nz} levels, depth 0\u2013{z[-1].item():.0f} m, dz = {dz} m")
print(f"Light at surface / bottom: {f_light[0].item():.3f} / {f_light[-1].item():.3f}")

## 2. Generate synthetic observations from a known $\mu(z)$

Run the coupled system forward with the true $\mu(z)$ profile and record the final $N$ and $P$ fields with small Gaussian noise. The MLP later sees only these two noisy profiles \u2014 it never sees the true $\mu(z)$ during training.

In [ ]:
N_obs, P_obs, mu_true = generate_coupled_observations(
    z=z, N0=N0, P0=P0, f_light=f_light,
    kappa=kappa, mortality=mortality, dz=dz, dt=dt, n_steps=n_steps,
    noise_std=0.005, seed=0,
)

print(f"N_obs range:  [{N_obs.min().item():.3f}, {N_obs.max().item():.3f}]")
print(f"P_obs range:  [{P_obs.min().item():.3f}, {P_obs.max().item():.3f}]")
print(f"mu_true range: [{mu_true.min().item():.3f}, {mu_true.max().item():.3f}]")

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].plot(N_obs, z, label="observed N", color="tab:blue")
axes[0].plot(N0, z, "--", color="tab:blue", alpha=0.4, label="initial N")
axes[0].invert_yaxis(); axes[0].set_xlabel("nutrient"); axes[0].set_ylabel("depth z (m)")
axes[0].set_title("Nutrient profile"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(P_obs, z, label="observed P", color="tab:green")
axes[1].plot(P0, z, "--", color="tab:green", alpha=0.4, label="initial P")
axes[1].invert_yaxis(); axes[1].set_xlabel("phytoplankton")
axes[1].set_title("Plankton profile"); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[2].plot(mu_true, z, label=r"true $\mu(z)$", color="tab:red")
axes[2].invert_yaxis(); axes[2].set_xlabel(r"uptake rate $\mu$")
axes[2].set_title(r"Ground-truth $\mu(z)$"); axes[2].legend(); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Train the MLP to recover $\mu(z)$ from both tracers

Each epoch:
1. MLP predicts $\mu_{\text{pred}}(z)$ from normalised depth.
2. Run the coupled simulator forward with $\mu_{\text{pred}}$.
3. MSE loss = MSE on $N$ + MSE on $P$ (the MLP must satisfy both).
4. Backpropagate through the 200-step coupled time integration into the MLP weights.

The coupled inverse problem is structurally harder than the single-tracer case: an error in $\mu$ propagates through the $N$\u2013$P$ coupling, so reaching the same tolerance requires more training.

In [ ]:
t0 = time.time()
mlp, losses = train_coupled_recovery(
    N_obs=N_obs, P_obs=P_obs, z=z, N0=N0, P0=P0, f_light=f_light,
    kappa=kappa, mortality=mortality, dz=dz, dt=dt, n_steps=n_steps,
    n_epochs=2500, lr=1e-2, device=device,
)
elapsed = time.time() - t0
print(f"Trained {len(losses)} epochs in {elapsed:.1f}s on {device}")
print(f"Loss: {losses[0]:.4e} \u2192 {losses[-1]:.4e} ({losses[0] / losses[-1]:.0f}\u00d7 reduction)")

## 4. Compare recovered $\mu(z)$ and tracer fits to ground truth

In [ ]:
z_norm = ((z - z.mean()) / z.std()).to(device)
features = z_norm.unsqueeze(-1)
with torch.no_grad():
    mu_pred_dev = mlp(features)
    N_pred_dev, P_pred_dev = integrate_coupled(
        N0.to(device), P0.to(device), mu_pred_dev,
        f_light.to(device), kappa, mortality, dz, dt, n_steps,
    )
mu_pred = mu_pred_dev.cpu()
N_pred = N_pred_dev.cpu()
P_pred = P_pred_dev.cpu()

rel_rmse_mu = (
    (mu_pred - mu_true).pow(2).mean().sqrt() / mu_true.mean()
).item()
rel_rmse_N = (
    (N_pred - N_obs).pow(2).mean().sqrt() / N_obs.mean()
).item()
rel_rmse_P = (
    (P_pred - P_obs).pow(2).mean().sqrt() / P_obs.mean()
).item()
print(f"Relative RMSE on mu(z): {rel_rmse_mu * 100:.2f}%")
print(f"Relative RMSE on N:    {rel_rmse_N * 100:.2f}%")
print(f"Relative RMSE on P:    {rel_rmse_P * 100:.2f}%")

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

axes[0].semilogy(losses)
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("MSE loss (log scale)")
axes[0].set_title("Training loss"); axes[0].grid(alpha=0.3)

axes[1].plot(mu_true, z, label="ground truth", linewidth=2, color="tab:red")
axes[1].plot(mu_pred, z, "--", label="MLP recovered", linewidth=2, color="tab:orange")
axes[1].invert_yaxis()
axes[1].set_xlabel(r"uptake rate $\mu(z)$"); axes[1].set_ylabel(r"depth $z$ (m)")
axes[1].set_title(f"Recovered $\\mu(z)$, rel RMSE = {rel_rmse_mu * 100:.1f}%")
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(N_obs, z, label="observed", linewidth=2, color="tab:blue")
axes[2].plot(N_pred, z, "--", label="recovered run", linewidth=2, color="tab:cyan")
axes[2].invert_yaxis()
axes[2].set_xlabel("nutrient"); axes[2].set_title(f"N fit, rel RMSE = {rel_rmse_N * 100:.1f}%")
axes[2].legend(); axes[2].grid(alpha=0.3)

axes[3].plot(P_obs, z, label="observed", linewidth=2, color="tab:green")
axes[3].plot(P_pred, z, "--", label="recovered run", linewidth=2, color="tab:olive")
axes[3].invert_yaxis()
axes[3].set_xlabel("phytoplankton"); axes[3].set_title(f"P fit, rel RMSE = {rel_rmse_P * 100:.1f}%")
axes[3].legend(); axes[3].grid(alpha=0.3)

plt.tight_layout(); plt.show()

## What this adds over Notebook 1

The single-tracer recovery in Notebook 1 confirmed PyTorch autograd flows through hand-coded numerical integration. This notebook confirms three additional things:

1. **The scaffold handles coupled tracers.** The simulator advances $N$ and $P$ together each step, autograd tracks both tracers through the full trajectory, and the gradient with respect to a single parameter $\mu(z)$ flows back through both coupled fields.
2. **The scaffold handles nonlinear reaction terms.** The uptake term $\mu \, P \, f_{\text{light}} \, N$ and the quadratic mortality $m P^2$ are both nonlinear; differentiation through them is stable across thousands of training steps.
3. **The inverse problem stays solvable when one parameter influences many tracers.** The MLP has to find a single $\mu(z)$ profile that simultaneously reproduces both $N$ and $P$ observations \u2014 the same structural problem that arises in real Darwin, where one parameter (e.g., iron scavenging rate) shapes many tracer fields through the biogeochemistry network.

The lit-zone restriction (0\u2013200 m) is a deliberate scoping choice: $\mu(z)$ is only identifiable where the uptake term is non-negligible. Real Darwin recovery will need the same kind of identifiability reasoning when picking which parameters to learn from which observations \u2014 a question to settle with the domain advisor.